In [5]:
import netCDF4 as nc
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from shapely.geometry import Point
import geopandas as gpd
from pyproj import Transformer
import os

**chdir**

In [6]:
os.chdir('/home/bill/GitHub/wps-research/data/bill')

In [7]:
f = './245/VNP14IMG.A2025245.1154.002.2025245203851.nc'

In [15]:
ds = nc.Dataset(f, 'r')

lat = np.array(ds['FP_latitude'][:])
lon = ds['FP_longitude'][:]
frp = ds['FP_power'][:]
conf = ds['FP_confidence'][:]

In [20]:
ds['FP_latitude'][:]

masked_array(data=[64.008156, 64.0034  , 64.008766, ..., 51.930813,
                   51.933502, 51.930744],
             mask=False,
       fill_value=1e+20,
            dtype=float32)

In [14]:
len(lon)

1139

In [10]:
for attr in ds.ncattrs():
    print(f"{attr}: {ds.getncattr(attr)}")

ds.close()

VNP02IMG: VNP02CCIMG.A2025245.1154.002.2025245203721.nc
VNP03IMG: VNP03IMG.A2025245.1154.002.2025245201453.nc
VNP02MOD: VNP02CCMOD.A2025245.1154.002.2025245203721.nc
VNP02GDC: VNP02GDC.A2025245.1154.002.2025245203239.nc
ProcessVersionNumber: 3.1.9
ExecutableCreationDate: Oct 31 2023
ExecutableCreationTime: 17:46:00
SystemID: Linux minion20371 5.4.0-1118-fips #128-Ubuntu SMP Wed Apr 2 02:11:38 UTC 2025 x86_64
Unagg_GRingLatitude: 50.778477,57.927536,78.048622,64.888306
Unagg_GRingLongitude: -123.385033,-170.288605,176.654297,-92.512238
NorthBoundingCoordinate: 79.11283111572266
SouthBoundingCoordinate: 50.77847671508789
EastBoundingCoordinate: -92.51223754882812
WestBoundingCoordinate: 176.654296875
Satellite: NPP
Day/Night/Both: Both
FirePix: 1139
DayPix: 2920815
LandPix: 2520872
NightPix: 38448785
RangeEndingTime: 12:00:00.00000
WaterPix: 399943
MissingPix: 3445
GlintPix: 0
CloudPix: 21209723
creator_email: modis-ops@lists.nasa.gov
PGENumber: 510
InputPointer: /MODAPSops6/archive/f203

In [11]:
transformer = Transformer.from_crs("EPSG:4326", "EPSG:32609", always_xy=True)
utm_x, utm_y = transformer.transform(lon, lat)

In [12]:
transformer = Transformer.from_crs("EPSG:4326", "EPSG:32609", always_xy=True)
utm_x, utm_y = transformer.transform(lon, lat)

In [13]:
df = pd.DataFrame({
    'latitude': lat,
    'longitude': lon,
    'utm_x': utm_x,
    'utm_y': utm_y,
    'FRP_MW': frp,
    'confidence': conf
})

In [14]:
geometry = [Point(xy) for xy in zip(utm_x, utm_y)]
gdf = gpd.GeoDataFrame(df, geometry=geometry, crs='EPSG:32609')

gdf.to_file('viirs_fires_utm9n.gpkg', driver='GPKG')